# Credit Risk Analysis - Feature Engineering

This notebook prepares the data for modeling by:
- Dropping useless and leakage features
- Creating binary target variable
- Adding temporal features
- Encoding categorical variables
- Handling missing values
- Scaling features
- Creating train/test split

## 1. Setup and Load Data

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/sample_100k_stratified.csv', low_memory=False)
print(f"Data loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

Data loaded: 100,000 rows x 152 columns


## 2. Create Binary Target Variable

In [4]:
default_statuses = ['Charged Off', 'Default', 'Late (31-120 days)', 'Late (16-30 days)']

df['is_default'] = df['loan_status'].apply(lambda x: 1 if x in default_statuses else 0)

print(f"Default rate: {df['is_default'].mean()*100:.2f}%")
print(f"Defaults: {df['is_default'].sum():,}")
print(f"Non-defaults: {(len(df) - df['is_default'].sum()):,}")

Default rate: 12.84%
Defaults: 12,842
Non-defaults: 87,158


## 3. Drop High-Missing Features (>95% missing)

In [5]:
# Calculate missing percentages
missing_pct = (df.isnull().sum() / len(df)) * 100
high_missing = missing_pct[missing_pct > 95].sort_values(ascending=False)

print(f"Features with >95% missing: {len(high_missing)}")
print("\nDropping these features:")
print(high_missing.head(10))

# Drop high-missing features
df = df.drop(columns=high_missing.index.tolist())
print(f"\n✅ Dataset shape after dropping: {df.shape}")

Features with >95% missing: 34

Dropping these features:
member_id                                     100.000
orig_projected_additional_accrued_interest     99.615
hardship_type                                  99.507
hardship_last_payment_amount                   99.507
hardship_payoff_balance_amount                 99.507
hardship_loan_status                           99.507
hardship_dpd                                   99.507
hardship_length                                99.507
payment_plan_start_date                        99.507
hardship_end_date                              99.507
dtype: float64

✅ Dataset shape after dropping: (100000, 119)


## 4. Drop Data Leakage Features

### Important Note on Data Leakage

**From our EDA, the "top risk factors" were:**
- `recoveries` (0.49 correlation)
- `collection_recovery_fee` (0.46 correlation)
- `total_rec_late_fee` (0.13 correlation)

**We cannot use these because:**

These features are **consequences of default, not predictors!** They represent events that happen **AFTER** a loan has already defaulted:

1. **Loan defaults** → borrower stops paying
2. **Collections begin** → lender tries to recover money
3. **Recoveries & fees** → these values get recorded

If we included these features, our model would have perfect hindsight:
- "If `recoveries` > 0, then the loan defaulted" 
- This is data leakage (using future information to predict the past)

**Features to drop (post-default data):**
- `recoveries`
- `collection_recovery_fee`
- `total_rec_late_fee`
- `last_pymnt_amnt`
- `last_pymnt_d`
- `total_rec_prncp`
- `total_rec_int`
- `total_pymnt`
- `total_pymnt_inv`
- `out_prncp`
- `out_prncp_inv`
- `last_fico_range_high`
- `last_fico_range_low`
- `last_credit_pull_d`

**What we will use (pre-loan or at-origination data):**
- `fico_range_low/high` (at origination)
- `int_rate` (assigned at origination)
- `grade` (assigned at origination)
- `annual_inc`, `dti`, `emp_length` (at application time)

In [ ]:
# List of leakage features to drop
leakage_features = [
    'recoveries', 'collection_recovery_fee', 'total_rec_late_fee',
    'last_pymnt_amnt', 'last_pymnt_d', 'total_rec_prncp', 'total_rec_int',
    'total_pymnt', 'total_pymnt_inv', 'out_prncp', 'out_prncp_inv',
    'last_fico_range_high', 'last_fico_range_low', 'last_credit_pull_d',
    'next_pymnt_d', 'funded_amnt', 'funded_amnt_inv', 'installment',
    'total_pymnt', 'issue_d'  # Keep issue_date separately for temporal features
]

# Drop leakage features that exist in the dataframe
leakage_to_drop = [f for f in leakage_features if f in df.columns]
print(f"Dropping {len(leakage_to_drop)} data leakage features...")

df = df.drop(columns=leakage_to_drop)
print(f"Dataset shape after removing leakage: {df.shape}")

## 5. Add Temporal Features

In [ ]:
# Extract temporal features from issue_d (before we dropped it)
# Reload to get issue_d
df_temp = pd.read_csv('../data/sample_100k_stratified.csv', usecols=['issue_d'], low_memory=False)
df['issue_d'] = df_temp['issue_d']

# Create temporal features
df['issue_date'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
df['issue_year'] = df['issue_date'].dt.year
df['issue_month'] = df['issue_date'].dt.month
df['is_post_2015'] = (df['issue_year'] >= 2015).astype(int)

# Drop the date columns (keep numeric features only)
df = df.drop(columns=['issue_d', 'issue_date'])

print("Temporal features added:")
print("   - issue_year")
print("   - issue_month")
print("   - is_post_2015 (binary)")

## 6. Drop Non-Predictive Features

In [ ]:
# Drop ID columns and non-predictive features
non_predictive = ['id', 'member_id', 'url', 'desc', 'title', 'zip_code', 
                 'emp_title', 'loan_status', 'pymnt_plan', 'policy_code']

to_drop = [f for f in non_predictive if f in df.columns]
df = df.drop(columns=to_drop)

print(f"Dropped {len(to_drop)} non-predictive features")
print(f"   Dataset shape: {df.shape}")

## 7. Feature Selection - Keep Important Features

In [ ]:
# Select key features for modeling
# Based on EDA insights and domain knowledge

key_features = [
    # Target
    'is_default',
    
    # Temporal
    'issue_year', 'issue_month', 'is_post_2015',
    
    # Loan characteristics (at origination)
    'loan_amnt', 'term', 'int_rate', 'grade', 'sub_grade',
    
    # Borrower characteristics
    'emp_length', 'home_ownership', 'annual_inc', 'verification_status',
    
    # Credit history (at origination)
    'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high',
    'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
    'total_acc', 'earliest_cr_line',
    
    # Loan purpose
    'purpose',
    
    # Application type
    'application_type'
]

# Keep only features that exist in the dataframe
available_features = [f for f in key_features if f in df.columns]
df_model = df[available_features].copy()

print(f"Selected {len(available_features)} key features for modeling")
print(f"   Dataset shape: {df_model.shape}")
print(f"\nFeatures: {available_features}")

## 8. Handle Missing Values

In [ ]:
# Check missing values in selected features
missing_summary = df_model.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)

print("Missing values in selected features:")
print(missing_summary)
print(f"\nTotal features with missing values: {len(missing_summary)}")

In [ ]:
# Strategy for handling missing values

# For numerical features: fill with median
numerical_cols = df_model.select_dtypes(include=['float64', 'int64']).columns
numerical_cols = [c for c in numerical_cols if c != 'is_default']

for col in numerical_cols:
    if df_model[col].isnull().sum() > 0:
        df_model[col].fillna(df_model[col].median(), inplace=True)

# For categorical features: fill with mode or 'Unknown'
categorical_cols = df_model.select_dtypes(include=['object']).columns

for col in categorical_cols:
    if df_model[col].isnull().sum() > 0:
        df_model[col].fillna('Unknown', inplace=True)

print("Missing values handled")
print(f"   Remaining missing values: {df_model.isnull().sum().sum()}")

## 9. Encode Categorical Variables

In [ ]:
# Get categorical columns
categorical_cols = df_model.select_dtypes(include=['object']).columns.tolist()

print(f"Categorical features to encode: {len(categorical_cols)}")
print(categorical_cols)

# One-hot encode categorical variables
df_encoded = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

print(f"\nEncoding complete")
print(f"   Dataset shape after encoding: {df_encoded.shape}")

## 10. Prepare Features and Target

In [ ]:
# Separate features and target
X = df_encoded.drop('is_default', axis=1)
y = df_encoded['is_default']

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")
print(f"\nDefault rate in full dataset: {y.mean()*100:.2f}%")

## 11. Train/Test Split

In [ ]:
# Split into train and test sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nDefault rate in train: {y_train.mean()*100:.2f}%")
print(f"Default rate in test: {y_test.mean()*100:.2f}%")

## 12. Feature Scaling

In [ ]:
# Scale features using StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames to preserve column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("Features scaled using StandardScaler")
print(f"   Train: {X_train_scaled.shape}")
print(f"   Test: {X_test_scaled.shape}")

## 13. Save Processed Data

In [ ]:
# Save processed datasets
import pickle

# Save train/test splits
X_train_scaled.to_csv('../data/X_train.csv', index=False)
X_test_scaled.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

# Save scaler for future use
with open('../data/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Processed data saved:")
print("   - data/X_train.csv")
print("   - data/X_test.csv")
print("   - data/y_train.csv")
print("   - data/y_test.csv")
print("   - data/scaler.pkl")

## 14. Summary

In [2]:
print("="*70)
print("FEATURE ENGINEERING SUMMARY")
print("="*70)

print(f"\n1. DATA PIPELINE:")
print(f"   Original features: 154")
print(f"   After dropping high-missing: ~121")
print(f"   After dropping leakage: ~90")
print(f"   After feature selection: {len(available_features)}")
print(f"   After encoding: {X.shape[1]} features")

print(f"\n2. TRAIN/TEST SPLIT:")
print(f"   Training samples: {len(X_train):,} (80%)")
print(f"   Test samples: {len(X_test):,} (20%)")
print(f"   Features: {X_train.shape[1]}")

print(f"\n3. CLASS BALANCE:")
print(f"   Train - Defaults: {y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"   Train - Non-defaults: {len(y_train)-y_train.sum():,} ({(1-y_train.mean())*100:.2f}%)")
print(f"   Test  - Defaults: {y_test.sum():,} ({y_test.mean()*100:.2f}%)")
print(f"   Test  - Non-defaults: {len(y_test)-y_test.sum():,} ({(1-y_test.mean())*100:.2f}%)")

print(f"\n4. KEY FEATURES INCLUDED:")
print(f"   ✓ Temporal: issue_year, issue_month, is_post_2015")
print(f"   ✓ FICO scores (at origination)")
print(f"   ✓ Loan grade & interest rate")
print(f"   ✓ Borrower income, DTI, employment")
print(f"   ✓ Credit history features")

print(f"\n5. DROPPED FEATURES:")
print(f"   ✗ High-missing (>95%): ~33 features")
print(f"   ✗ Data leakage (post-default): ~15 features")
print(f"   ✗ Non-predictive (IDs, URLs): ~10 features")

print("\n" + "="*70)
print("Data is ready for modeling.")
print("="*70)

FEATURE ENGINEERING SUMMARY

1. DATA PIPELINE:
   Original features: 154
   After dropping high-missing: ~121
   After dropping leakage: ~90


NameError: name 'available_features' is not defined